# NB_lab_test_trends
Builds `adb_caresync.gold.lab_test_trends` per the STTM *Silver to Gold* (sheet `lab_test_trends`).
* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `silver.lab_results` grouped by `test_name`, year and month of `test_date`
* **Joins:** LEFT JOIN `silver.calendar` on the first day of the month's `date_key`
* **Window:** `LAG(abnormal_rate_pct)` by test, ordered by year, month (month-over-month trend)
* **Load type:** FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT)

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends
COMMENT 'Gold: one row per test type per month with abnormal/critical rates and month-over-month trend'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                         AS year,
    month(test_date)                        AS month,
    COUNT(*)                                AS total_tests,
    COUNT_IF(result_status = 'Abnormal')    AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')    AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  r.test_name,
  CAST(r.year AS INT)                                                   AS year,
  CAST(r.month AS INT)                                                  AS month,
  c.month_name,
  c.quarter_name,
  CAST(r.total_tests AS INT)                                            AS total_tests,
  CAST(r.abnormal_test_count AS INT)                                    AS abnormal_test_count,
  CAST(r.critical_test_count AS INT)                                    AS critical_test_count,
  r.abnormal_rate_pct,
  r.critical_rate_pct,
  CAST(r.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                  AS prev_month_abnormal_rate_pct,
  CAST(r.abnormal_rate_pct - r.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag r
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(r.year, r.month, 1), 'yyyyMMdd') AS INT)

In [0]:
updated_df=spark.read.table("adb_caresync.gold.lab_test_trends")
record_count=updated_df.count()
dbutils.notebook.exit(str(record_count))
